In [1]:
import sqlite3
import os

db_path = "../04_Data_Warehouse/breast_cancer_warehouse.db"

if not os.path.exists(db_path):
    raise FileNotFoundError(f"Database not found: {db_path}")

conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""")

print("Connected to the warehouse successfully!")
print("\nAvailable tables:")

for row in cursor.fetchall():
    print("-", row[0])

cursor.execute("SELECT COUNT(*) FROM Fact_Breast_Cancer")
print("\nTotal fact records:", cursor.fetchone()[0])

Connected to the warehouse successfully!

Available tables:
- Dim_Clinical
- Dim_Diagnosis
- Dim_Location
- Dim_Patient
- Dim_Tumor
- Fact_Breast_Cancer

Total fact records: 272


In [2]:
query = """
SELECT
    d.Class AS Diagnosis_Class,
    COUNT(*) AS Total_Records,
    SUM(f.Patient_Count) AS Total_Patient_Count
FROM Fact_Breast_Cancer f
JOIN Dim_Diagnosis d
    ON f.Diagnosis_Key = d.Diagnosis_Key
GROUP BY d.Class
ORDER BY d.Class;
"""

cursor.execute(query)
results = cursor.fetchall()

print("OLAP Roll-up: Records by Diagnosis Class")
print("-" * 48)

for row in results:
    print(
        f"Class: {row[0]} | "
        f"Records: {row[1]} | "
        f"Patient Count: {row[2]}"
    )

OLAP Roll-up: Records by Diagnosis Class
------------------------------------------------
Class: 0 | Records: 191 | Patient Count: 191
Class: 1 | Records: 81 | Patient Count: 81


In [3]:
query = """
SELECT
    d.Class AS Diagnosis_Class,
    t.Deg_Malig AS Degree_of_Malignancy,
    COUNT(*) AS Total_Records
FROM Fact_Breast_Cancer f
JOIN Dim_Diagnosis d
    ON f.Diagnosis_Key = d.Diagnosis_Key
JOIN Dim_Tumor t
    ON f.Tumor_Key = t.Tumor_Key
GROUP BY d.Class, t.Deg_Malig
ORDER BY d.Class, t.Deg_Malig;
"""

cursor.execute(query)
results = cursor.fetchall()

print("OLAP Drill-down: Diagnosis by Degree of Malignancy")
print("-" * 60)

for row in results:
    print(
        f"Class: {row[0]} | "
        f"Malignancy: {row[1]} | "
        f"Records: {row[2]}"
    )

OLAP Drill-down: Diagnosis by Degree of Malignancy
------------------------------------------------------------
Class: 0 | Malignancy: 1 | Records: 54
Class: 0 | Malignancy: 2 | Records: 98
Class: 0 | Malignancy: 3 | Records: 39
Class: 1 | Malignancy: 1 | Records: 12
Class: 1 | Malignancy: 2 | Records: 26
Class: 1 | Malignancy: 3 | Records: 43


In [4]:
query = """
SELECT
    d.Class AS Diagnosis_Class,
    COUNT(*) AS Total_Records,
    AVG(t.Tumor_Size) AS Average_Tumor_Size,
    AVG(t.Inv_Nodes) AS Average_Involved_Nodes,
    AVG(t.Deg_Malig) AS Average_Degree_of_Malignancy
FROM Fact_Breast_Cancer f
JOIN Dim_Diagnosis d
    ON f.Diagnosis_Key = d.Diagnosis_Key
JOIN Dim_Tumor t
    ON f.Tumor_Key = t.Tumor_Key
WHERE d.Class = 1
GROUP BY d.Class;
"""

cursor.execute(query)
results = cursor.fetchall()

print("OLAP Slice: Diagnosis Class 1")
print("-" * 45)

for row in results:
    print("Diagnosis Class:", row[0])
    print("Total Records:", row[1])
    print("Average Tumor Size:", round(row[2], 2))
    print("Average Involved Nodes:", round(row[3], 2))
    print("Average Degree of Malignancy:", round(row[4], 2))

OLAP Slice: Diagnosis Class 1
---------------------------------------------
Diagnosis Class: 1
Total Records: 81
Average Tumor Size: 29.04
Average Involved Nodes: 4.11
Average Degree of Malignancy: 2.38


In [5]:
query = """
SELECT
    d.Class AS Diagnosis_Class,
    t.Deg_Malig AS Degree_of_Malignancy,
    t.Tumor_Size,
    COUNT(*) AS Total_Records,
    SUM(f.Patient_Count) AS Total_Patient_Count
FROM Fact_Breast_Cancer f
JOIN Dim_Diagnosis d
    ON f.Diagnosis_Key = d.Diagnosis_Key
JOIN Dim_Tumor t
    ON f.Tumor_Key = t.Tumor_Key
WHERE d.Class = 1
  AND t.Deg_Malig = 3
  AND t.Tumor_Size >= 30
GROUP BY d.Class, t.Deg_Malig, t.Tumor_Size
ORDER BY t.Tumor_Size;
"""

cursor.execute(query)
results = cursor.fetchall()

print("OLAP Dice: Class 1, Malignancy 3, Tumor Size >= 30")
print("-" * 60)

if results:
    for row in results:
        print(
            f"Class: {row[0]} | "
            f"Malignancy: {row[1]} | "
            f"Tumor Size: {row[2]} | "
            f"Records: {row[3]} | "
            f"Patient Count: {row[4]}"
        )
else:
    print("No records match these conditions.")

OLAP Dice: Class 1, Malignancy 3, Tumor Size >= 30
------------------------------------------------------------
Class: 1 | Malignancy: 3 | Tumor Size: 32.0 | Records: 16 | Patient Count: 16
Class: 1 | Malignancy: 3 | Tumor Size: 37.0 | Records: 4 | Patient Count: 4
Class: 1 | Malignancy: 3 | Tumor Size: 42.0 | Records: 2 | Patient Count: 2
Class: 1 | Malignancy: 3 | Tumor Size: 52.0 | Records: 1 | Patient Count: 1


In [6]:
query = """
SELECT
    d.Class AS Diagnosis_Class,
    COUNT(*) AS Total_Records,
    ROUND(AVG(t.Tumor_Size), 2) AS Avg_Tumor_Size,
    ROUND(AVG(t.Inv_Nodes), 2) AS Avg_Involved_Nodes,
    ROUND(AVG(t.Deg_Malig), 2) AS Avg_Degree_of_Malignancy,
    MIN(t.Tumor_Size) AS Min_Tumor_Size,
    MAX(t.Tumor_Size) AS Max_Tumor_Size
FROM Fact_Breast_Cancer f
JOIN Dim_Diagnosis d
    ON f.Diagnosis_Key = d.Diagnosis_Key
JOIN Dim_Tumor t
    ON f.Tumor_Key = t.Tumor_Key
GROUP BY d.Class
ORDER BY d.Class;
"""

cursor.execute(query)
results = cursor.fetchall()

print("OLAP Aggregation: Tumor Statistics by Diagnosis Class")
print("-" * 65)

for row in results:
    print(f"\nDiagnosis Class: {row[0]}")
    print(f"Total Records: {row[1]}")
    print(f"Average Tumor Size: {row[2]}")
    print(f"Average Involved Nodes: {row[3]}")
    print(f"Average Degree of Malignancy: {row[4]}")
    print(f"Minimum Tumor Size: {row[5]}")
    print(f"Maximum Tumor Size: {row[6]}")

OLAP Aggregation: Tumor Statistics by Diagnosis Class
-----------------------------------------------------------------

Diagnosis Class: 0
Total Records: 191
Average Tumor Size: 25.56
Average Involved Nodes: 2.01
Average Degree of Malignancy: 1.92
Minimum Tumor Size: 2.0
Maximum Tumor Size: 52.0

Diagnosis Class: 1
Total Records: 81
Average Tumor Size: 29.04
Average Involved Nodes: 4.11
Average Degree of Malignancy: 2.38
Minimum Tumor Size: 2.0
Maximum Tumor Size: 52.0


In [7]:
# Final OLAP validation

cursor.execute("""
    SELECT COUNT(*), SUM(Patient_Count),
           COUNT(DISTINCT Fact_Key)
    FROM Fact_Breast_Cancer
""")

total_records, total_patients, unique_keys = cursor.fetchone()

cursor.execute("PRAGMA foreign_key_check")
fk_violations = cursor.fetchall()

print("Total fact records:", total_records)
print("Total patient count:", total_patients)
print("Unique fact keys:", unique_keys)
print("Foreign-key violations:", fk_violations)

if (
    total_records == 272
    and total_patients == 272
    and unique_keys == 272
    and not fk_violations
):
    print("\nOLAP VALIDATION PASSED")
else:
    print("\nCHECK REQUIRED")

Total fact records: 272
Total patient count: 272
Unique fact keys: 272
Foreign-key violations: []

OLAP VALIDATION PASSED
